# Simulação de confrontos diretos e de temporada regular
* Serão capturados os últimos elos registrados para cada time;
* Será testado o resultado de um confronto direto;
* Será simulado um torneio sem monte carlo, e com monte carlo.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

In [2]:
ROOT = Path.cwd().parent
df_elo = pd.read_csv(ROOT /'historico_nbb' / 'data' / 'historico_elo.csv')
nbb_26 = pd.read_csv(ROOT /'historico_nbb' / 'data' / 'calendario_26_nbb.csv')

In [3]:
df_elo.head()

,Season,Time,Elo
0,2016,Bauru,1489.551101
1,2016,Flamengo,1510.448899
2,2016,Pinheiros,1489.293738
3,2016,Vitória,1510.706262
4,2016,Minas,1508.842695


In [4]:
nbb_26.head()

,Season,Casa,Visitante
0,2026,Mogi,Corinthians
1,2026,Caxias do Sul Basquete,Cruzeiro
2,2026,UNIFACISA,Basket Osasco
3,2026,Flamengo,Bauru
4,2026,Franca,São José


In [5]:
df_elo['Time'].unique() # Vendo a lista dos times que estão no histórico de elo

<StringArray>
[                 'Bauru',               'Flamengo',              'Pinheiros',
                'Vitória',                  'Minas', 'Caxias do Sul Basquete',
                 'Franca',         'Macaé Basquete',      'Basquete Cearense',
          'Vasco da Gama',               'Brasília',           'Campo Mourão',
          'L. Sorocabana',                   'Mogi',             'Paulistano',
       'Joinville / AABJ',               'Botafogo',               'São José',
            'Corinthians',            'VipTech CMB',              'UNIFACISA',
          'Pato Basquete',              'São Paulo',       'Cerrado Basquete',
      'União Corinthians',              'Rio Claro',               'Cruzeiro',
          'Basket Osasco']
Length: 28, dtype: str

In [6]:
nbb_26['Casa'].unique() # Vendo a lista dos times que estão no calendário da temporada 2026

<StringArray>
[                  'Mogi', 'Caxias do Sul Basquete',              'UNIFACISA',
               'Flamengo',                 'Franca',      'União Corinthians',
      'Basquete Cearense',                  'Bauru',          'Pato Basquete',
               'Brasília',            'Corinthians',                  'Minas',
               'Cruzeiro',              'Pinheiros',             'Paulistano',
               'São José',               'Botafogo',          'Basket Osasco']
Length: 18, dtype: str

In [7]:
# ===========================================================
# Coletando o último elo de cada time para a temporada 2026
# ===========================================================
elo_26 = df_elo[df_elo['Season'] == 2025].groupby('Time').last().reset_index()[['Time', 'Elo']]

elo_26

,Time,Elo
0,Basket Osasco,1290.285361
1,Basquete Cearense,1254.020408
2,Bauru,1586.872178
3,Botafogo,1359.344906
4,Brasília,1760.525917
5,Caxias do Sul Basquete,1499.711879
6,Corinthians,1762.702554
7,Cruzeiro,1444.275302
8,Flamengo,1708.818785
9,Franca,1910.205789


In [8]:
# ===========================================================
# Simulando uma partida entre equipes selecionadas
# ===========================================================
bsb = elo_26[elo_26['Time'] == 'Brasília'].iloc[0]['Elo']
fla = elo_26[elo_26['Time'] == 'Flamengo'].iloc[0]['Elo']

prob_bsb = 1 / (1 + 10 ** ((fla - bsb) / 400))

result = np.random.choice(['Brasília', 'Flamengo'], p=[prob_bsb, 1 - prob_bsb])

print(f"Probabilidade de vitória do Brasília: {prob_bsb:.2%}")
print(f"Probabilidade de vitória do Flamengo: {1 - prob_bsb:.2%}")
print(f"Resultado da partida: {result}")

Probabilidade de vitória do Brasília: 57.39%
Probabilidade de vitória do Flamengo: 42.61%
Resultado da partida: Brasília


In [9]:
# ====================================================================================
# Simulando uma partida entre equipes selecionadas com vantagem de casa
# ====================================================================================
casa = 60
# Supondo que o Brasília jogue em casa
bsb = elo_26[elo_26['Time'] == 'Brasília'].iloc[0]['Elo']
fla = elo_26[elo_26['Time'] == 'Flamengo'].iloc[0]['Elo']
vtg =  (bsb + casa) - fla

prob_bsb = 1 / (1 + 10 ** ((-vtg) / 400))

result = np.random.choice(['Brasília', 'Flamengo'], p=[prob_bsb, 1 - prob_bsb])

print(f"Probabilidade de vitória do Brasília: {prob_bsb:.2%}")
print(f"Probabilidade de vitória do Flamengo: {1 - prob_bsb:.2%}")
print(f"Resultado da partida: {result}")

Probabilidade de vitória do Brasília: 65.54%
Probabilidade de vitória do Flamengo: 34.46%
Resultado da partida: Brasília


In [10]:
# ====================================================================================
# Simulando uma partida entre equipes selecionadas com vantagem de casa várias vezes
# ====================================================================================
casa = 60
# Supondo que o Brasília jogue em casa
bsb = elo_26[elo_26['Time'] == 'Brasília'].iloc[0]['Elo']
fla = elo_26[elo_26['Time'] == 'Flamengo'].iloc[0]['Elo']
vtg =  (bsb + casa) - fla

prob_bsb = 1 / (1 + 10 ** ((-vtg) / 400))

result = []
for _ in range(5):  # Simula 5 partidas
    result.append(np.random.choice(['Brasília', 'Flamengo'], p=[prob_bsb, 1 - prob_bsb]))


print(f"Resultado da partida: {result}")

Resultado da partida: [np.str_('Flamengo'), np.str_('Brasília'), np.str_('Brasília'), np.str_('Flamengo'), np.str_('Brasília')]


In [11]:
# ====================================================================================
# Simulando uma temporada da NBB 2026 com base no histórico de elo
# ====================================================================================
for row in nbb_26.itertuples():
    casa = row.Casa
    fora = row.Visitante
    
    elo_casa = elo_26[elo_26['Time'] == casa].iloc[0]['Elo']
    elo_fora = elo_26[elo_26['Time'] == fora].iloc[0]['Elo']
    vtg = (elo_casa + 60) - elo_fora  # Adicionando vantagem de casa de 60 pontos
    
    prob_casa = 1 / (1 + 10 ** ((-vtg) / 400))
    
    result = np.random.choice([casa, fora], p=[prob_casa, 1 - prob_casa])
    
    print(f"Jogo: {casa} vs {fora} | Resultado: {result} | Probabilidade de vitória do {casa}: {prob_casa:.2%}")

Jogo: Mogi vs Corinthians | Resultado: Corinthians | Probabilidade de vitória do Mogi: 49.64%
Jogo: Caxias do Sul Basquete vs Cruzeiro | Resultado: Caxias do Sul Basquete | Probabilidade de vitória do Caxias do Sul Basquete: 66.03%
Jogo: UNIFACISA vs Basket Osasco | Resultado: UNIFACISA | Probabilidade de vitória do UNIFACISA: 88.70%
Jogo: Flamengo vs Bauru | Resultado: Flamengo | Probabilidade de vitória do Flamengo: 74.03%
Jogo: Franca vs São José | Resultado: Franca | Probabilidade de vitória do Franca: 92.60%
Jogo: União Corinthians vs Cruzeiro | Resultado: União Corinthians | Probabilidade de vitória do União Corinthians: 67.95%
Jogo: UNIFACISA vs Pinheiros | Resultado: Pinheiros | Probabilidade de vitória do UNIFACISA: 26.22%
Jogo: Caxias do Sul Basquete vs Minas | Resultado: Minas | Probabilidade de vitória do Caxias do Sul Basquete: 30.62%
Jogo: Basquete Cearense vs Basket Osasco | Resultado: Basquete Cearense | Probabilidade de vitória do Basquete Cearense: 53.41%
Jogo: Flamen

In [ ]:
# ====================================================================================
# Simulando a temporada com mini monte carlo
# ====================================================================================
historico_posicoes = []
for i in range(100):  # Simulando 10.000 temporadas
    tabela = {time: 0 for time in elo_26['Time']}  # Inicializando a tabela de vitórias

    for row in nbb_26.itertuples():
        casa = row.Casa
        fora = row.Visitante
        
        elo_casa = elo_26[elo_26['Time'] == casa].iloc[0]['Elo']
        elo_fora = elo_26[elo_26['Time'] == fora].iloc[0]['Elo']
        vtg = (elo_casa + 60) - elo_fora  # Adicionando vantagem de casa de 60 pontos
        
        prob_casa = 1 / (1 + 10 ** ((-vtg) / 400))
        
        vencedor = np.random.choice([casa, fora], p=[prob_casa, 1 - prob_casa])
        tabela[vencedor] += 1

        classificacao = sorted(
            tabela.items(),
            key=lambda x: x[1],
            reverse=True
        )
    historico_posicoes.append(classificacao)

In [13]:
# ====================================================================================
# Observando os resultados da simulação
# ====================================================================================

# Analisando as probabilidades do Brasília
hist_bsb = []

for classificacao in historico_posicoes:
    for posicao, (time, vitorias) in enumerate(classificacao, start=1):
        if time == 'Brasília':
            hist_bsb.append(posicao)
            break


hist_bsb = np.array(hist_bsb)

bsb_campeao = np.sum(hist_bsb == 1) / len(hist_bsb)
bsb_top4 = np.sum(hist_bsb <= 4) / len(hist_bsb)
bsb_playoffs = np.sum(hist_bsb <= 8) / len(hist_bsb) 
print(f"Probabilidade do Brasília ser campeão: {bsb_campeao:.2%}")
print(f"Probabilidade do Brasília ficar entre os 4 primeiros: {bsb_top4:.2%}")
print(f"Probabilidade do Brasília ficar entre os 8 primeiros (playoffs): {bsb_playoffs:.2%}")

Probabilidade do Brasília ser campeão: 17.00%
Probabilidade do Brasília ficar entre os 4 primeiros: 69.00%
Probabilidade do Brasília ficar entre os 8 primeiros (playoffs): 94.00%


In [19]:
# ====================================================================================
# Simulando a temporada com monte carlo com otimizações
# ====================================================================================
elo_dict = dict(zip(elo_26['Time'], elo_26['Elo']))  # Criando um dicionário para acesso rápido aos elos

historico_posicoes = []
for i in range(10000):  # Simulando 10.000 temporadas
    tabela = {time: 0 for time in elo_26['Time']}  # Inicializando a tabela de vitórias

    for row in nbb_26.itertuples():
        casa = row.Casa
        fora = row.Visitante
        
        elo_casa = elo_dict[casa]
        elo_fora = elo_dict[fora]
        vtg = (elo_casa + 60) - elo_fora  # Adicionando vantagem de casa de 60 pontos
        
        prob_casa = 1 / (1 + 10 ** ((-vtg) / 400))
        
        vencedor = np.random.choice([casa, fora], p=[prob_casa, 1 - prob_casa])
        tabela[vencedor] += 1

        classificacao = sorted(
            tabela.items(),
            key=lambda x: x[1],
            reverse=True
        )
        time_alvo = 'Corinthians'  # Substitua pelo time que você deseja analisar
        
        for posicao, (time, vitorias) in enumerate(classificacao, start=1):
            if time == time_alvo:
                historico_posicoes.append(posicao)
                break
        

In [20]:
# ====================================================================================
# Observando os resultados da simulação
# ====================================================================================

hist_cor = np.array(historico_posicoes)

cor_campeao = np.mean(hist_cor == 1)
cor_top4 = np.mean(hist_cor <= 4)
cor_playoffs = np.mean(hist_cor <= 8)
print(f"Probabilidade do Corinthians ser campeão: {cor_campeao:.2%}")
print(f"Probabilidade do Corinthians ficar entre os 4 primeiros: {cor_top4:.2%}")
print(f"Probabilidade do Corinthians ficar entre os 8 primeiros (playoffs): {cor_playoffs:.2%}")

Probabilidade do Corinthians ser campeão: 6.36%
Probabilidade do Corinthians ficar entre os 4 primeiros: 35.48%
Probabilidade do Corinthians ficar entre os 8 primeiros (playoffs): 68.43%
